# ⚡ PJM Hourly Electricity Load Forecasting
**Mini project – Time Series Forecasting + Streamlit deployment**

Dataset: `PJMW_MW_Hourly.xlsx` (PJM West region, hourly demand in MW, 2002-2018)

| Stage | What we do |
|---|---|
| 1 | Load data and check quality (sorting, duplicates, missing hours, outliers) |
| 2 | Clean the data |
| 3 | Exploratory data analysis (hourly / weekly / monthly / yearly / seasonal patterns) |
| 4 | Feature engineering (lags, rolling statistics, calendar, holidays) |
| 5 | Chronological train / test split |
| 6 | Models: Naive, Seasonal-Naive, Linear Regression, Random Forest, XGBoost, ANN, LSTM |
| 7 | Evaluation (MAE, RMSE, MAPE, R²) |
| 8 | Honest multi-day back-test (recursive forecasting) vs Holt-Winters |
| 9 | Save model + generate the Streamlit app files |

**How to run:** `Runtime → Run all`. When asked, upload `PJMW_MW_Hourly.xlsx`.

## 0. Setup

In [ ]:
!pip -q install xgboost openpyxl statsmodels

In [ ]:
# Upload the dataset (Colab). If the file is already in the folder this is skipped.
import os
if not os.path.exists("PJMW_MW_Hourly.xlsx"):
    try:
        from google.colab import files
        print("Please upload PJMW_MW_Hourly.xlsx")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Put PJMW_MW_Hourly.xlsx next to this notebook.")
print("Dataset found:", os.path.exists("PJMW_MW_Hourly.xlsx"))

### Shared code file
The next cell writes `forecast_utils.py`. **The notebook (training) and the Streamlit app (deployment) both import it**, so the features used at training time are *exactly* the features built at prediction time. (In the earlier version the app built different weekday columns than the model was trained on, so the weekday information was silently lost.)

In [ ]:
%%writefile forecast_utils.py
"""
forecast_utils.py
-----------------
Shared code used by BOTH the Colab notebook (training) and the Streamlit app
(deployment).  Using one file guarantees that the features the model saw during
training are built in exactly the same way when the app makes a forecast.
"""
import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

TARGET = "PJMW_MW"
LAGS = [1, 2, 3, 24, 48, 168, 336, 720]
MAX_HISTORY = max(LAGS)          # 720 hours = 30 days of history needed

FEATURE_COLS = (
    [f"lag_{l}" for l in LAGS]
    + ["roll_mean_24", "roll_std_24", "roll_mean_168"]
    + ["Hour", "Day_of_Week", "Month", "Year", "Is_Weekend", "Is_Holiday"]
)

# US federal holidays (pandas built-in calendar, no extra package needed)
_HOLIDAYS = set(
    USFederalHolidayCalendar().holidays(start="2000-01-01", end="2045-12-31")
)


# ------------------------------------------------------------------
# 1. DATA LOADING + CLEANING
# ------------------------------------------------------------------
def clean_series(raw: pd.DataFrame, outlier_below: float = 1000.0):
    """
    Turn the raw PJMW table into a clean, strictly hourly pd.Series.

    Steps
      1. parse Datetime / numeric target, drop unreadable rows
      2. sort chronologically (the raw file is NOT sorted)
      3. average duplicated timestamps (daylight-saving 'fall back' hour)
      4. reindex to a complete hourly range (daylight-saving 'spring forward'
         hours are missing in the raw file)
      5. treat impossible values (< outlier_below MW, e.g. the 487 MW reading)
         as missing
      6. fill every gap by time interpolation
    Returns (series, report_dict)
    """
    df = raw[["Datetime", TARGET]].copy()
    df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce")
    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
    n_raw = len(df)
    df = df.dropna(subset=["Datetime"])

    was_sorted = bool(df["Datetime"].is_monotonic_increasing)
    n_dupes = int(df["Datetime"].duplicated().sum())

    s = (df.groupby("Datetime")[TARGET].mean().sort_index())
    full_index = pd.date_range(s.index.min(), s.index.max(), freq="h")
    n_missing_hours = int(len(full_index) - len(s))
    s = s.reindex(full_index)

    outliers = s < outlier_below
    n_outliers = int(outliers.sum())
    s[outliers] = np.nan

    n_to_fill = int(s.isna().sum())
    s = s.interpolate(method="time", limit_direction="both")
    s.index.name = "Datetime"
    s.name = TARGET

    report = dict(
        raw_rows=n_raw,
        was_sorted=was_sorted,
        duplicate_timestamps=n_dupes,
        missing_hours=n_missing_hours,
        outliers_replaced=n_outliers,
        values_filled=n_to_fill,
        clean_rows=int(len(s)),
        start=str(s.index.min()),
        end=str(s.index.max()),
    )
    return s, report


def load_clean_series(path: str):
    """Load the clean CSV written by the notebook, or the raw xlsx/csv file."""
    if path.lower().endswith(".xlsx"):
        raw = pd.read_excel(path)
    else:
        raw = pd.read_csv(path)
    return clean_series(raw)


# ------------------------------------------------------------------
# 2. FEATURES FOR TRAINING  (vectorised, whole series at once)
# ------------------------------------------------------------------
def _calendar_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    out = pd.DataFrame(index=index)
    out["Hour"] = index.hour
    out["Day_of_Week"] = index.dayofweek
    out["Month"] = index.month
    out["Year"] = index.year
    out["Is_Weekend"] = (index.dayofweek >= 5).astype(int)
    out["Is_Holiday"] = index.normalize().isin(list(_HOLIDAYS)).astype(int)
    return out


def make_feature_frame(series: pd.Series) -> pd.DataFrame:
    """Lag, rolling and calendar features for every hour of `series`."""
    X = pd.DataFrame(index=series.index)
    for l in LAGS:
        X[f"lag_{l}"] = series.shift(l)
    past = series.shift(1)                     # never include the current hour
    X["roll_mean_24"] = past.rolling(24).mean()
    X["roll_std_24"] = past.rolling(24).std()  # ddof=1, same as build_row()
    X["roll_mean_168"] = past.rolling(168).mean()
    X = X.join(_calendar_features(series.index))
    return X[FEATURE_COLS]


# ------------------------------------------------------------------
# 3. FEATURES FOR ONE FUTURE HOUR  (used in recursive forecasting / app)
# ------------------------------------------------------------------
def build_row(buf: np.ndarray, ts: pd.Timestamp) -> np.ndarray:
    """
    buf : array whose LAST element is the load of hour (ts - 1h); length >= 720
    ts  : the hour we want to predict
    """
    row = [buf[-l] for l in LAGS]
    row += [buf[-24:].mean(), buf[-24:].std(ddof=1), buf[-168:].mean()]
    row += [
        ts.hour,
        ts.dayofweek,
        ts.month,
        ts.year,
        int(ts.dayofweek >= 5),
        int(ts.normalize() in _HOLIDAYS),
    ]
    return np.asarray(row, dtype=float)


def recursive_forecast(model, history_values, next_timestamp, horizon: int):
    """
    Forecast `horizon` hours, one hour at a time.  Each prediction is appended
    to the history so it can feed the lag features of the following hour.

    history_values : 1-D array of the most recent actual loads (>= 720 values)
    next_timestamp : timestamp of the FIRST hour to predict
    """
    hist = np.asarray(history_values, dtype=float)
    if len(hist) < MAX_HISTORY:
        raise ValueError(f"Need at least {MAX_HISTORY} hours of history, got {len(hist)}.")
    buf = hist[-MAX_HISTORY:].copy()
    ts = pd.Timestamp(next_timestamp)
    preds = np.empty(horizon)
    for i in range(horizon):
        x = pd.DataFrame([build_row(buf, ts + pd.Timedelta(hours=i))], columns=FEATURE_COLS)
        p = float(model.predict(x)[0])
        p = max(p, 0.0)
        preds[i] = p
        buf = np.append(buf[1:], p)
    return preds


# ------------------------------------------------------------------
# 4. METRICS
# ------------------------------------------------------------------
def regression_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_true - y_pred
    mae = float(np.mean(np.abs(err)))
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mape = float(np.mean(np.abs(err) / np.maximum(np.abs(y_true), 1e-9)) * 100)
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return dict(MAE=mae, RMSE=rmse, MAPE=mape, R2=r2)

In [ ]:
import warnings, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from xgboost import XGBRegressor

from forecast_utils import (TARGET, LAGS, FEATURE_COLS, MAX_HISTORY, clean_series,
                            make_feature_frame, recursive_forecast, regression_metrics)

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Load data and check quality

In [ ]:
raw = pd.read_excel("PJMW_MW_Hourly.xlsx")
print("Shape:", raw.shape)
raw.head()

In [ ]:
raw["Datetime"] = pd.to_datetime(raw["Datetime"])
print(raw.dtypes, "\n")
print("Missing values      :", raw.isna().sum().sum())
print("Sorted by time?     :", raw["Datetime"].is_monotonic_increasing)
print("Duplicate timestamps:", raw["Datetime"].duplicated().sum())

tmp = raw.drop_duplicates("Datetime").sort_values("Datetime")
expected = len(pd.date_range(tmp["Datetime"].min(), tmp["Datetime"].max(), freq="h"))
print("Expected hourly rows:", expected, "| actual unique rows:", len(tmp), "| missing hours:", expected - len(tmp))
print("Date range:", tmp["Datetime"].min(), "->", tmp["Datetime"].max())
raw[TARGET].describe()

In [ ]:
print("Duplicate timestamps (daylight-saving 'fall back' hour):")
display(raw[raw["Datetime"].duplicated(keep=False)].sort_values("Datetime"))

print("\nLowest values:")
display(raw.nsmallest(3, TARGET))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(x=raw[TARGET], ax=ax[0]); ax[0].set_title("Box plot of raw load")
ax[1].plot(tmp.set_index("Datetime")[TARGET]["2003-05-27":"2003-05-31"]); ax[1].set_title("The 487 MW reading (2003-05-29)")
plt.show()

**Findings:** the raw file is *not* in chronological order, has 4 duplicated timestamps (daylight-saving "fall-back" hour), about 30 missing hours ("spring-forward" hour) and one impossible reading of **487 MW** (normal load is ~4,000-7,000 MW).
These matter because lag features such as `lag_24` assume that every row is exactly one hour after the previous one.

## 2. Data cleaning

In [ ]:
series, report = clean_series(raw)
display(pd.Series(report, name="value").to_frame())

assert series.index.to_series().diff().dropna().eq(pd.Timedelta("1h")).all(), "index is not strictly hourly"
print("\nClean series is strictly hourly:", len(series), "rows")

In [ ]:
# Save the clean data - the Streamlit app loads this (fast) instead of the Excel file
series.reset_index().to_csv("pjm_clean.csv", index=False)
print("Saved pjm_clean.csv")

Cleaning rules: sort → average duplicate timestamps → insert the missing hours → treat values below 1,000 MW as invalid → fill every gap by **time interpolation** (better than the previous approach of filling with the hour-0 average).

## 3. Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(15, 8))
ax[0].plot(series, lw=.4); ax[0].set_title("PJM West hourly load 2002-2018 (cleaned)"); ax[0].set_ylabel("MW")
last = series["2018-06-01":"2018-06-28"]
ax[1].plot(last); ax[1].set_title("Four weeks zoomed in: daily AND weekly rhythm"); ax[1].set_ylabel("MW")
plt.tight_layout(); plt.show()

In [ ]:
eda = pd.DataFrame({"load": series})
eda["Hour"] = eda.index.hour
eda["Day_of_Week"] = eda.index.day_name()
eda["Month"] = eda.index.month
eda["Year"] = eda.index.year
eda["Season"] = eda["Month"].map(lambda m: "Winter" if m in (12, 1, 2) else "Summer" if m in (6, 7, 8) else "Other")

fig, ax = plt.subplots(2, 2, figsize=(15, 9))
eda.groupby("Hour")["load"].mean().plot(marker="o", ax=ax[0, 0], title="Average load by hour of day")
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
eda.groupby("Day_of_Week")["load"].mean().reindex(order).plot.bar(ax=ax[0, 1], title="Average load by weekday", rot=30)
eda.groupby("Month")["load"].mean().plot(marker="o", ax=ax[1, 0], title="Average load by month")
eda.groupby("Year")["load"].mean().plot(marker="o", ax=ax[1, 1], title="Average load by year")
for a in ax.ravel(): a.set_ylabel("MW")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=eda[eda.Season != "Other"], x="Hour", y="load", hue="Season", errorbar=None, marker="o", ax=ax[0])
ax[0].set_title("Hourly profile: Summer vs Winter")
sns.boxplot(data=eda[eda.Season != "Other"], x="Season", y="load", ax=ax[1]); ax[1].set_title("Load distribution by season")
plt.show()

pivot = eda.pivot_table(index="Day_of_Week", columns="Hour", values="load", aggfunc="mean").reindex(order)
plt.figure(figsize=(14, 4)); sns.heatmap(pivot, cmap="YlOrRd"); plt.title("Average load: weekday x hour"); plt.show()

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(15, 10))
series.resample("D").mean().plot(ax=ax[0], title="Daily average", lw=.7)
series.resample("W").mean().plot(ax=ax[1], title="Weekly average")
series.resample("ME").mean().plot(ax=ax[2], title="Monthly average")
plt.tight_layout(); plt.show()

# Trend: rolling means
last_year = series["2017-08-01":"2018-08-03"]
plt.figure(figsize=(15, 4))
plt.plot(last_year, lw=.3, alpha=.5, label="hourly")
plt.plot(last_year.rolling(24).mean(), label="24-hour rolling mean")
plt.plot(last_year.rolling(24 * 30).mean(), label="30-day rolling mean", lw=2)
plt.legend(); plt.title("Trend via rolling averages"); plt.show()

In [ ]:
# Seasonality: autocorrelation shows spikes at 24 h (daily) and 168 h (weekly)
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(series["2016-01-01":], lags=24 * 8, ax=plt.subplots(figsize=(14, 4))[1])
plt.title("Autocorrelation - peaks at 24, 48, ... and 168 hours"); plt.xlabel("Lag (hours)"); plt.show()

**Patterns found**
* **Daily (24 h):** low overnight, peak in the afternoon/evening.
* **Weekly (168 h):** weekdays are higher than weekends.
* **Yearly:** winter and summer peaks (heating / cooling).
* These repeating cycles are why we use lags of 24 and 168 hours as features.

## 4. Feature engineering
Features for predicting the load at hour *t* (all use only information available **before** *t*):

* **Lags:** load 1, 2, 3, 24, 48, 168, 336 and 720 hours ago
* **Rolling statistics:** mean/std of the previous 24 h, mean of the previous 168 h
* **Calendar:** hour, weekday, month, year, weekend flag, **US federal holiday** flag

In [ ]:
X_all = make_feature_frame(series)
data = X_all.join(series).dropna()          # first 720 hours have no 720-h lag
X, y = data[FEATURE_COLS], data[TARGET]
print("Feature matrix:", X.shape)
print("Features:", FEATURE_COLS)
X.head()

In [ ]:
# Correlation with the target
corr = data.corr()[TARGET].drop(TARGET).sort_values()
corr.plot.barh(figsize=(8, 6), title="Correlation of each feature with load"); plt.show()

In [ ]:
# Safety check used for deployment: does the one-hour-at-a-time builder used by the app
# produce EXACTLY the same features as the vectorised builder used for training?
from forecast_utils import build_row
for r in [1000, 50000, 100000, 143000]:
    ts = series.index[r]
    assert np.allclose(build_row(series.values[:r], ts), X_all.iloc[r].values), f"mismatch at row {r}"
print("Training features == serving features  OK")

## 5. Chronological train / test split
Random shuffling would leak the future into training, so the **oldest 80 %** is used for training and the **newest 20 %** for testing.

In [ ]:
n_train = int(len(data) * 0.8)
X_train, X_test = X.iloc[:n_train], X.iloc[n_train:]
y_train, y_test = y.iloc[:n_train], y.iloc[n_train:]
pos = series.index.get_indexer(data.index)      # position of every modelled hour inside `series`
print("Train:", X_train.index.min(), "->", X_train.index.max(), f"({len(X_train):,} rows)")
print("Test :", X_test.index.min(),  "->", X_test.index.max(),  f"({len(X_test):,} rows)")

results, preds = [], {}
def evaluate(name, y_true, y_pred):
    m = regression_metrics(y_true, y_pred)
    results.append({"Model": name, **m}); preds[name] = np.asarray(y_pred)
    print(f"{name:<22} MAE {m['MAE']:8.2f} | RMSE {m['RMSE']:8.2f} | MAPE {m['MAPE']:5.2f}% | R2 {m['R2']:.4f}")

## 6. Models (1-hour-ahead prediction on the test set)

### 6.1 Baselines

In [ ]:
evaluate("Naive (last hour)", y_test, X_test["lag_1"])
evaluate("Seasonal naive (last week)", y_test, X_test["lag_168"])

### 6.2 Linear Regression

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
evaluate("Linear Regression", y_test, lr.predict(X_test))

### 6.3 Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
evaluate("Random Forest", y_test, rf.predict(X_test))

### 6.4 XGBoost

In [ ]:
def new_xgb():
    return XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=8, subsample=0.8,
                        colsample_bytree=0.8, objective="reg:squarederror",
                        tree_method="hist", random_state=42, n_jobs=-1)

xgb = new_xgb().fit(X_train, y_train)
evaluate("XGBoost", y_test, xgb.predict(X_test))

pd.Series(xgb.feature_importances_, index=FEATURE_COLS).sort_values().plot.barh(
    figsize=(8, 6), title="XGBoost feature importance"); plt.show()

### 6.5 ANN (feed-forward neural network on the same features)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.random.set_seed(42)

sc_x = StandardScaler().fit(X_train)
sc_y = StandardScaler().fit(y_train.values.reshape(-1, 1))

ann = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"), layers.Dropout(0.1),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),
])
ann.compile(optimizer="adam", loss="mse")
history_ann = ann.fit(
    sc_x.transform(X_train), sc_y.transform(y_train.values.reshape(-1, 1)),
    epochs=50, batch_size=512, validation_split=0.1, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)])

p = sc_y.inverse_transform(ann.predict(sc_x.transform(X_test), verbose=0)).ravel()
evaluate("ANN", y_test, p)
print("Epochs run:", len(history_ann.history["loss"]))

plt.plot(history_ann.history["loss"], label="train"); plt.plot(history_ann.history["val_loss"], label="validation")
plt.title("ANN loss"); plt.xlabel("epoch"); plt.legend(); plt.show()

### 6.6 LSTM (sequence model, input = previous 24 hours)

In [ ]:
SEQ = 24
train_end_pos = pos[n_train - 1] + 1

sc_l = MinMaxScaler().fit(series.values[:train_end_pos].reshape(-1, 1))   # fit on TRAIN only
scaled = sc_l.transform(series.values.reshape(-1, 1)).ravel()
windows = np.lib.stride_tricks.sliding_window_view(scaled, SEQ)            # windows[j] = scaled[j:j+SEQ]
X_seq = windows[pos - SEQ][..., None]                                      # same target hours as the tabular models
y_seq = scaled[pos]

Xl_tr, Xl_te = X_seq[:n_train], X_seq[n_train:]
yl_tr = y_seq[:n_train]

lstm = keras.Sequential([
    layers.Input(shape=(SEQ, 1)),
    layers.LSTM(64, return_sequences=True), layers.Dropout(0.2),
    layers.LSTM(32), layers.Dropout(0.2),
    layers.Dense(16, activation="relu"), layers.Dense(1),
])
lstm.compile(optimizer="adam", loss="mse")
history_lstm = lstm.fit(Xl_tr, yl_tr, epochs=30, batch_size=256, validation_split=0.1, verbose=0,
                        callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)])

p = sc_l.inverse_transform(lstm.predict(Xl_te, verbose=0)).ravel()
evaluate("LSTM", y_test, p)
print("Epochs run:", len(history_lstm.history["loss"]))

plt.plot(history_lstm.history["loss"], label="train"); plt.plot(history_lstm.history["val_loss"], label="validation")
plt.title("LSTM loss"); plt.xlabel("epoch"); plt.legend(); plt.show()

## 7. Model comparison

In [ ]:
comparison = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(comparison.round(3))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=comparison, y="Model", x="RMSE", ax=ax[0], color="#f5a524"); ax[0].set_title("RMSE (lower is better)")
sns.barplot(data=comparison, y="Model", x="MAPE", ax=ax[1], color="#5b6b8c"); ax[1].set_title("MAPE % (lower is better)")
plt.tight_layout(); plt.show()

In [ ]:
best_ml = comparison.iloc[0]["Model"]
print("Best model on 1-hour-ahead RMSE:", best_ml)

# Actual vs predicted for the last two weeks of the test period
sl = slice(-24 * 14, None)
plt.figure(figsize=(15, 5))
plt.plot(y_test.index[sl], y_test.values[sl], label="Actual", lw=2)
plt.plot(y_test.index[sl], preds["XGBoost"][sl], label="XGBoost", alpha=.9)
plt.title("XGBoost vs actual - last 14 days of test set"); plt.ylabel("MW"); plt.legend(); plt.show()

res_x = y_test.values - preds["XGBoost"]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(res_x, bins=60, ax=ax[0]); ax[0].set_title("XGBoost residuals (MW)")
ax[1].scatter(y_test.values, preds["XGBoost"], s=2, alpha=.3); ax[1].set_title("Actual vs predicted")
ax[1].set_xlabel("Actual"); ax[1].set_ylabel("Predicted"); plt.show()

> **Important:** the table above is *one-hour-ahead* accuracy – the model always sees the true previous hour. The deployed app forecasts **many hours ahead**, feeding its own predictions back in (recursive forecasting), which is harder. The next section measures that honestly.

## 8. Multi-day (recursive) back-test
We pick 10 start dates spread over the test period, forecast the next **7 days** from each one using only older data, and compare with real load.
Competitors: **Seasonal naive** (repeat last week) and **Holt-Winters** (exponential smoothing, 24-h seasonality).

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

def holt_winters_forecast(history, h):
    fit = ExponentialSmoothing(history, seasonal="add", seasonal_periods=24,
                               initialization_method="estimated").fit()
    return fit.forecast(h)

H = 168
starts = np.linspace(pos[n_train], pos[-1] - H, 10).astype(int)
store = {"XGBoost (recursive)": [], "Seasonal naive": [], "Holt-Winters": []}
truth = []
t0 = time.time()
for p0 in starts:
    ts0 = series.index[p0]
    hist = series.values[:p0]
    truth.append(series.values[p0:p0 + H])
    store["XGBoost (recursive)"].append(recursive_forecast(xgb, hist, ts0, H))
    store["Seasonal naive"].append(hist[-168:])
    store["Holt-Winters"].append(holt_winters_forecast(hist[-24 * 60:], H))
print(f"back-test finished in {time.time() - t0:.0f}s")

truth = np.array(truth)
rows = []
for h in (24, 72, 168):
    for name, arr in store.items():
        m = regression_metrics(truth[:, :h].ravel(), np.array(arr)[:, :h].ravel())
        rows.append({"Method": name, "Horizon (hours)": h, "MAE": m["MAE"], "RMSE": m["RMSE"], "MAPE": m["MAPE"]})
horizon_table = pd.DataFrame(rows)
display(horizon_table.round(2))

In [ ]:
# Error growth with the forecast horizon
err = np.abs(truth - np.array(store["XGBoost (recursive)"])).mean(axis=0)
plt.figure(figsize=(12, 4)); plt.plot(err)
plt.title("XGBoost recursive forecast: mean absolute error vs hours ahead")
plt.xlabel("hours ahead"); plt.ylabel("MAE (MW)"); plt.show()

# One example window
k = 4
plt.figure(figsize=(14, 5))
idx = series.index[starts[k]: starts[k] + H]
plt.plot(idx, truth[k], label="Actual", lw=2)
plt.plot(idx, store["XGBoost (recursive)"][k], label="XGBoost (recursive)")
plt.plot(idx, store["Holt-Winters"][k], label="Holt-Winters", alpha=.7)
plt.plot(idx, store["Seasonal naive"][k], label="Seasonal naive", alpha=.5)
plt.title("7-day forecast example"); plt.ylabel("MW"); plt.legend(); plt.show()

## 9. Final model for deployment
For the app we retrain XGBoost on **all** available data (standard practice - the test split was only needed to measure accuracy), then save it in XGBoost's portable **JSON** format. JSON works across operating systems and XGBoost versions, unlike `.pkl` files (which caused trouble when moving from Windows to macOS).

In [ ]:
final_model = new_xgb().fit(X, y)
final_model.save_model("xgb_model.json")

# reload test - proves the file works
check = XGBRegressor(); check.load_model("xgb_model.json")
assert np.allclose(check.predict(X.tail(100)), final_model.predict(X.tail(100)))
print("Saved xgb_model.json and verified it reloads correctly")

metrics = {
    "model_comparison": comparison.to_dict("records"),
    "horizon_backtest": horizon_table.to_dict("records"),
    "data_report": report,
    "deployed_model": "XGBoost (retrained on all data)",
}
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=float)
print("Saved metrics.json")

In [ ]:
# Demo: 30-day future forecast with the final model
future = recursive_forecast(final_model, series.values, series.index[-1] + pd.Timedelta(hours=1), 24 * 30)
fidx = pd.date_range(series.index[-1] + pd.Timedelta(hours=1), periods=len(future), freq="h")
plt.figure(figsize=(15, 5))
plt.plot(series.tail(24 * 14), label="Last 14 days (actual)")
plt.plot(fidx, future, label="30-day forecast", color="#f5a524")
plt.legend(); plt.title("30-day forecast beyond the end of the dataset"); plt.ylabel("MW"); plt.show()

## 10. Streamlit app files
The next cells write `app.py`, `requirements.txt`, the theme config and a README. `forecast_utils.py`, `xgb_model.json`, `pjm_clean.csv` and `metrics.json` were created above.

In [ ]:
%%writefile app.py
"""
PJM electricity load forecast - Streamlit app
Run:  streamlit run app.py
"""
import json
import os

import altair as alt
import numpy as np
import pandas as pd
import streamlit as st

from forecast_utils import (
    FEATURE_COLS, MAX_HISTORY, TARGET,
    clean_series, recursive_forecast, regression_metrics,
)

st.set_page_config(page_title="PJM Load Forecast", page_icon="⚡",
                   layout="wide", initial_sidebar_state="expanded")

AMBER, SLATE, RED, TEAL = "#f5a524", "#5b6b8c", "#ff6b6b", "#2dd4bf"
INK, PANEL, MUTED = "#111a2e", "#111a2e", "#6b7794"

st.markdown(f"""
<style>
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600&family=Space+Grotesk:wght@500;600;700&display=swap');
html, body, .stApp {{ font-family: 'IBM Plex Sans', sans-serif; }}
h1, h2, h3 {{ font-family: 'Space Grotesk', sans-serif !important; }}
.block-container {{ padding-top: 1.6rem; max-width: 1280px; }}
#MainMenu, footer {{ visibility: hidden; }}
.title {{ font-family:'Space Grotesk',sans-serif; font-size:2.6rem; font-weight:700; line-height:1.05; color:{INK}; }}
.title span {{ color:{AMBER}; }}
.lede {{ margin-top:10px; color:{MUTED}; font-size:1.02rem; max-width:520px; line-height:1.55; }}
.pulse-label {{ color:{MUTED}; font-size:.86rem; margin-bottom:4px; }}
.pulse-value {{ font-family:'Space Grotesk',sans-serif; font-size:1.8rem; font-weight:600; color:{INK}; }}
.pulse-value small {{ font-size:.95rem; color:{MUTED}; font-weight:500; }}
.strip {{ display:grid; grid-auto-flow:column; grid-auto-columns:1fr; background:{PANEL};
          border-radius:10px; margin:10px 0 18px 0; }}
.strip .cell {{ padding:16px 20px; border-right:1px solid #1f2a44; }}
.strip .cell:last-child {{ border-right:none; }}
.strip .lab {{ color:#93a1bd; font-size:.86rem; }}
.strip .val {{ font-family:'Space Grotesk',sans-serif; font-size:1.5rem; font-weight:600; color:#f4f7fd; margin-top:2px; }}
.strip .sub {{ color:#7f8fb0; font-size:.8rem; margin-top:2px; }}
.strip .cell.hot .val {{ color:{AMBER}; }}
.sec {{ font-family:'Space Grotesk',sans-serif; font-size:1.25rem; font-weight:600; color:{INK}; margin:22px 0 4px 0; }}
.sec-note {{ color:{MUTED}; font-size:.9rem; margin-bottom:8px; }}
.banner {{ border-left:3px solid {AMBER}; background:{PANEL}; padding:12px 16px; border-radius:0 8px 8px 0; color:#e6ebf5; margin:6px 0 14px 0; }}
.banner b {{ color:{AMBER}; }}
.side-card {{ background:{PANEL}; border-radius:8px; padding:10px 12px; margin:10px 0 14px 0; font-size:.92rem; color:#e6ebf5; }}
.side-card b {{ color:{AMBER}; }}
div.stButton > button[kind="primary"] {{ background:{AMBER}; color:#0b1220; border:1px solid {AMBER};
          font-weight:600; border-radius:8px; height:3rem; }}
</style>
""", unsafe_allow_html=True)


# ---------------------------------------------------------------- helpers
def wide(fn, *args, **kwargs):
    """Call a Streamlit function full-width on old AND new Streamlit versions."""
    try:
        return fn(*args, width="stretch", **kwargs)
    except Exception:
        return fn(*args, use_container_width=True, **kwargs)


def strip(items):
    cells = "".join(
        f'<div class="cell{" hot" if hot else ""}"><div class="lab">{a}</div>'
        f'<div class="val">{b}</div><div class="sub">{c}</div></div>'
        for a, b, c, hot in items)
    st.markdown(f'<div class="strip">{cells}</div>', unsafe_allow_html=True)


def section(title, note=None):
    html = f'<div class="sec">{title}</div>'
    if note:
        html += f'<div class="sec-note">{note}</div>'
    st.markdown(html, unsafe_allow_html=True)


def sparkline(values, w=420, h=70):
    v = np.asarray(values, float)
    xs = np.linspace(2, w - 6, len(v))
    ys = h - 8 - (v - v.min()) / (v.max() - v.min() + 1e-9) * (h - 16)
    pts = " ".join(f"{x:.1f},{y:.1f}" for x, y in zip(xs, ys))
    area = f"2,{h} {pts} {w - 6},{h}"
    return (f'<svg viewBox="0 0 {w} {h}" width="100%" height="{h}" preserveAspectRatio="none">'
            f'<polygon points="{area}" fill="{AMBER}" fill-opacity="0.12"/>'
            f'<polyline points="{pts}" fill="none" stroke="{AMBER}" stroke-width="1.6"/>'
            f'<circle cx="{xs[-1]:.1f}" cy="{ys[-1]:.1f}" r="3.5" fill="{AMBER}"/></svg>')


def style(chart, height=380):
    return (chart.properties(height=height)
            .configure_axis(labelColor=MUTED, titleColor=MUTED, gridColor="#e6eaf2", labelFontSize=12)
            .configure_legend(orient="top", labelFontSize=12, title=None))


# ---------------------------------------------------------------- loading
HERE = os.path.dirname(os.path.abspath(__file__))


def path(name):
    return os.path.join(HERE, name)


@st.cache_resource(show_spinner="Loading XGBoost model...")
def load_model():
    try:
        from xgboost import XGBRegressor
    except ImportError:
        st.error("xgboost is not installed. Run:  pip install xgboost\n\n"
                 "On macOS also run:  brew install libomp")
        st.stop()
    except Exception as e:  # e.g. libomp missing on macOS
        st.error(f"xgboost could not be loaded: {e}\n\nOn macOS run:  brew install libomp")
        st.stop()
    model = XGBRegressor()
    model.load_model(path("xgb_model.json"))
    return model


@st.cache_data(show_spinner="Loading and cleaning dataset...")
def load_data():
    if os.path.exists(path("pjm_clean.csv")):
        d = pd.read_csv(path("pjm_clean.csv"), parse_dates=["Datetime"])
        return d.set_index("Datetime")[TARGET].asfreq("h")
    raw = pd.read_excel(path("PJMW_MW_Hourly.xlsx"))
    s, _ = clean_series(raw)
    return s


@st.cache_data
def load_metrics():
    p = path("metrics.json")
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return None


try:
    model = load_model()
    series = load_data()
except FileNotFoundError as e:
    st.error(f"File not found: {e.filename}. Keep app.py, forecast_utils.py, xgb_model.json "
             "and pjm_clean.csv (or PJMW_MW_Hourly.xlsx) in the same folder.")
    st.stop()

metrics = load_metrics()
first_ts, last_ts = series.index.min(), series.index.max()
last_load = float(series.iloc[-1])

# ---------------------------------------------------------------- sidebar
st.sidebar.markdown("## Forecast settings")
days = st.sidebar.slider("Forecast period (days)", 1, 30, 7)
hours = days * 24
st.sidebar.markdown(f'<div class="side-card"><b>{days} day(s)</b> = <b>{hours}</b> hourly predictions</div>',
                    unsafe_allow_html=True)

st.sidebar.markdown("### Forecast start")
after_end = st.sidebar.checkbox("Start right after the last dataset value", value=True)
if after_end:
    start = last_ts + pd.Timedelta(hours=1)
else:
    min_date = (first_ts + pd.Timedelta(hours=MAX_HISTORY)).date()
    max_date = (last_ts + pd.Timedelta(days=30)).date()
    d = st.sidebar.date_input("Select date", value=last_ts.date() - pd.Timedelta(days=60),
                              min_value=min_date, max_value=max_date)
    t = st.sidebar.time_input("Select time", value=pd.Timestamp("00:00").time())
    start = pd.Timestamp(f"{d} {t}").floor("h")
    start = max(start, first_ts + pd.Timedelta(hours=MAX_HISTORY))
st.sidebar.markdown(f'<div class="side-card">Forecast starts at<br><b>{start:%Y-%m-%d %H:%M}</b></div>',
                    unsafe_allow_html=True)
if not after_end and start <= last_ts:
    st.sidebar.info("Start is inside the dataset, so the forecast is compared with the real load (back-test).")

with st.sidebar.expander("About the model"):
    st.markdown("- XGBoost Regressor\n- Recursive hourly forecasting\n"
                "- Lags: 1, 2, 3, 24, 48, 168, 336, 720 h\n- Rolling mean / std\n"
                "- Hour, weekday, month, year, weekend, US holiday")

# ---------------------------------------------------------------- header
c1, c2 = st.columns([1.25, 1])
with c1:
    st.markdown('<div class="title">PJM electricity<br>load <span>forecast</span></div>'
                '<div class="lede">Hourly demand forecasts for up to 30 days ahead, generated by a '
                'trained XGBoost model on PJM West data.</div>', unsafe_allow_html=True)
with c2:
    st.markdown(f'<div class="pulse-label">Last 7 days of actual load</div>{sparkline(series.tail(168).values)}'
                f'<div class="pulse-value">{last_load:,.0f} <small>MW at {last_ts:%Y-%m-%d %H:%M}</small></div>',
                unsafe_allow_html=True)

strip([("Historical data start", f"{first_ts:%Y-%m-%d}", "First record", False),
       ("Last available data", f"{last_ts:%Y-%m-%d %H:%M}", "Latest record", False),
       ("Hourly records", f"{len(series):,}", "After cleaning", False),
       ("Last actual load", f"{last_load:,.0f} MW", "Latest reading", True)])

# ---------------------------------------------------------------- run
if wide(st.button, f"Generate {days}-day forecast", type="primary"):
    with st.spinner(f"Generating {hours} hourly predictions..."):
        if start <= last_ts:                        # back-test inside the data
            pos = series.index.get_loc(start)
            history = series.values[:pos]
            preds = recursive_forecast(model, history, start, hours)
            idx = pd.date_range(start, periods=hours, freq="h")
            hist_df = series.iloc[max(0, pos - 168):pos]
            actual = series.reindex(idx)
        else:                                       # true future forecast
            gap = int((start - last_ts) / pd.Timedelta(hours=1)) - 1
            full = recursive_forecast(model, series.values, last_ts + pd.Timedelta(hours=1), gap + hours)
            preds = full[gap:]
            idx = pd.date_range(start, periods=hours, freq="h")
            hist_df = series.tail(168)
            actual = pd.Series(np.nan, index=idx)

    forecast_df = pd.DataFrame({"Prediction_Number": np.arange(1, hours + 1),
                                "Datetime": idx, "Predicted_MW": preds})
    daily_df = (forecast_df.set_index("Datetime")["Predicted_MW"].resample("D")
                .agg(Average_MW="mean", Maximum_MW="max", Minimum_MW="min", Total_MWh="sum").reset_index())
    st.session_state["result"] = dict(forecast_df=forecast_df, daily_df=daily_df, hist_df=hist_df,
                                      actual=actual, start=start, days=days, hours=hours)

res = st.session_state.get("result")
if res is None:
    st.info("**1.** Pick a period in the sidebar  →  **2.** choose the start  →  **3.** click *Generate*.")
    if metrics:
        st.caption("Scroll down after generating a forecast to see model accuracy.")
    st.stop()

fdf, ddf, hist_df, actual = res["forecast_df"], res["daily_df"], res["hist_df"], res["actual"]
preds = fdf["Predicted_MW"].values
times = fdf["Datetime"]
imax, imin = int(np.argmax(preds)), int(np.argmin(preds))

st.markdown(f'<div class="banner">Forecast ready: <b>{res["hours"]} hourly predictions</b> for '
            f'{res["days"]} day(s), starting {res["start"]:%Y-%m-%d %H:%M}.</div>', unsafe_allow_html=True)
strip([("Average load", f"{preds.mean():,.0f} MW", f"Across {res['hours']} hours", False),
       ("Peak load", f"{preds.max():,.0f} MW", f"{times.iloc[imax]:%Y-%m-%d %H:%M}", True),
       ("Minimum load", f"{preds.min():,.0f} MW", f"{times.iloc[imin]:%Y-%m-%d %H:%M}", False),
       ("Total energy", f"{preds.sum() / 1000:,.1f} GWh", "Sum of hourly load", False)])

has_actual = actual.notna().all()
if has_actual:
    m = regression_metrics(actual.values, preds)
    strip([("Back-test MAE", f"{m['MAE']:,.0f} MW", "vs real load", False),
           ("Back-test RMSE", f"{m['RMSE']:,.0f} MW", "vs real load", False),
           ("Back-test MAPE", f"{m['MAPE']:.2f} %", "vs real load", True),
           ("Back-test R²", f"{m['R2']:.3f}", "vs real load", False)])

tabs = st.tabs(["Forecast", "Daily view", "Hourly data", "Export", "Model performance"])

# ---- Forecast tab
with tabs[0]:
    section("Hourly load forecast", "Slate = last 7 days of actual load, amber = forecast"
            + (", teal = real load during the forecast window." if has_actual else "."))
    parts = [hist_df.rename(TARGET).reset_index().rename(columns={TARGET: "Load_MW"}).assign(Series="Recent actual"),
             fdf[["Datetime", "Predicted_MW"]].rename(columns={"Predicted_MW": "Load_MW"}).assign(Series="Forecast")]
    domain, colors = ["Recent actual", "Forecast"], [SLATE, AMBER]
    if has_actual:
        parts.append(pd.DataFrame({"Datetime": actual.index, "Load_MW": actual.values}).assign(Series="Actual (back-test)"))
        domain.append("Actual (back-test)"); colors.append(TEAL)
    comb = pd.concat(parts, ignore_index=True)
    base = alt.Chart(comb).encode(
        x=alt.X("Datetime:T", title=None),
        y=alt.Y("Load_MW:Q", title="Load (MW)", scale=alt.Scale(zero=False)),
        color=alt.Color("Series:N", scale=alt.Scale(domain=domain, range=colors)))
    layers = [base.mark_line(strokeWidth=2.2),
              base.mark_circle(size=50, opacity=0).encode(
                  tooltip=["Series:N", alt.Tooltip("Datetime:T", format="%Y-%m-%d %H:%M"),
                           alt.Tooltip("Load_MW:Q", format=",.1f")])]
    marks = pd.DataFrame({"Datetime": [times.iloc[imax], times.iloc[imin]], "Load_MW": [preds[imax], preds[imin]],
                          "Label": ["Peak", "Low"]})
    layers += [alt.Chart(marks).mark_circle(size=130).encode(
        x="Datetime:T", y="Load_MW:Q", color=alt.Color("Label:N", scale=alt.Scale(domain=["Peak", "Low"], range=[RED, TEAL]), legend=None))]
    layers += [alt.Chart(pd.DataFrame({"Datetime": [res["start"]]})).mark_rule(strokeDash=[5, 4], color=MUTED).encode(x="Datetime:T")]
    wide(st.altair_chart, style(alt.layer(*layers).interactive(), 400), theme=None)
    st.caption("Forecasts are recursive: every predicted hour becomes part of the history used to build "
               "the lag features of the next hour, so uncertainty grows with the horizon.")

# ---- Daily tab
with tabs[1]:
    section("Daily average load", "Bars = daily average, thin line = daily min to max.")
    bars = alt.Chart(ddf).mark_bar(color=AMBER, cornerRadiusTopLeft=4, cornerRadiusTopRight=4).encode(
        x=alt.X("Datetime:T", timeUnit="yearmonthdate", title=None),
        y=alt.Y("Average_MW:Q", title="Load (MW)", scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("Datetime:T", format="%Y-%m-%d"), alt.Tooltip("Average_MW:Q", format=",.1f"),
                 alt.Tooltip("Maximum_MW:Q", format=",.1f"), alt.Tooltip("Minimum_MW:Q", format=",.1f")])
    rng = alt.Chart(ddf).mark_rule(color=INK, strokeWidth=1.5).encode(
        x=alt.X("Datetime:T", timeUnit="yearmonthdate"), y="Minimum_MW:Q", y2="Maximum_MW:Q")
    wide(st.altair_chart, style(alt.layer(bars, rng), 320), theme=None)
    if res["days"] >= 2:
        section("Load pattern by hour", "Each row is a day, each column an hour. Brighter = higher demand.")
        heat_df = fdf.assign(Date=fdf["Datetime"].dt.strftime("%b %d"), Hour=fdf["Datetime"].dt.hour)
        heat = alt.Chart(heat_df).mark_rect().encode(
            x=alt.X("Hour:O", title="Hour of day", axis=alt.Axis(labelAngle=0)),
            y=alt.Y("Date:O", sort=None, title=None),
            color=alt.Color("Predicted_MW:Q", scale=alt.Scale(scheme="inferno"), legend=alt.Legend(title="MW", orient="right")),
            tooltip=["Date:O", "Hour:O", alt.Tooltip("Predicted_MW:Q", format=",.1f")])
        wide(st.altair_chart, style(heat, max(180, 26 * res["days"] + 60)), theme=None)
    section("Daily summary table")
    wide(st.dataframe, ddf, hide_index=True)

# ---- Hourly tab
with tabs[2]:
    section(f"Hourly predictions ({res['hours']} rows)")
    show = fdf.copy()
    if has_actual:
        show["Actual_MW"] = actual.values
        show["Abs_Error_MW"] = (show["Actual_MW"] - show["Predicted_MW"]).abs()
    wide(st.dataframe, show, hide_index=True)

# ---- Export tab
with tabs[3]:
    section("Download results", "Both files open directly in Excel.")
    a, b = st.columns(2)
    with a:
        wide(st.download_button, "Download hourly forecast (CSV)", fdf.to_csv(index=False).encode(),
             file_name=f"pjm_load_forecast_{res['days']}_days.csv", mime="text/csv")
    with b:
        wide(st.download_button, "Download daily summary (CSV)", ddf.to_csv(index=False).encode(),
             file_name=f"pjm_daily_summary_{res['days']}_days.csv", mime="text/csv")

# ---- Model performance tab
with tabs[4]:
    if not metrics:
        st.info("metrics.json not found. Run the Colab notebook to generate it.")
    else:
        section("Model comparison (1-hour-ahead, chronological test set)")
        wide(st.dataframe, pd.DataFrame(metrics["model_comparison"]).round(3), hide_index=True)
        if metrics.get("horizon_backtest"):
            section("Recursive back-test (what this app really does)",
                    "Average error of XGBoost when it forecasts many hours ahead, across several test-period windows.")
            wide(st.dataframe, pd.DataFrame(metrics["horizon_backtest"]).round(3), hide_index=True)

In [ ]:
%%writefile requirements.txt
# ---- Streamlit app (deployment) ----
streamlit>=1.40
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
xgboost>=2.0
joblib>=1.3
openpyxl>=3.1
altair>=5.0

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)

In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#f5a524"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#f4f6fb"
textColor = "#111a2e"

[server]
headless = true

In [ ]:
%%writefile README.md
# PJM Hourly Electricity Load Forecasting (Mini Project)

Time-series forecasting of hourly PJM West demand (MW) with a Streamlit web app.

## Files
| File | Purpose |
|---|---|
| `PJM_Load_Forecasting.ipynb` | Colab notebook: cleaning, EDA, features, 7 models, evaluation, saving |
| `forecast_utils.py` | Cleaning + feature code shared by notebook and app |
| `app.py` | Streamlit app |
| `xgb_model.json` | Trained XGBoost model (portable JSON, no pickle problems) |
| `pjm_clean.csv` | Cleaned dataset used by the app |
| `metrics.json` | Model scores shown in the app |
| `requirements.txt` | Minimal dependencies for the app |

## Run on macOS (Apple Silicon or Intel)
```bash
# 1. one-time: OpenMP library that XGBoost needs on macOS
brew install libomp

# 2. create a clean environment (Python 3.10 - 3.12)
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt

# 3. run
streamlit run app.py
```
Opens at http://localhost:8501

## Deploy on Streamlit Community Cloud
Push `app.py`, `forecast_utils.py`, `xgb_model.json`, `pjm_clean.csv`, `metrics.json`,
`requirements.txt` and `.streamlit/config.toml` to a GitHub repo, then create the app at share.streamlit.io.

In [ ]:
# Bundle everything and download it (this zip is all you need on your Mac)
import shutil
os.makedirs("pjm_project/.streamlit", exist_ok=True)
for f in ["app.py", "forecast_utils.py", "requirements.txt", "xgb_model.json", "pjm_clean.csv", "metrics.json", "README.md"]:
    shutil.copy(f, "pjm_project/" + f)
shutil.copy(".streamlit/config.toml", "pjm_project/.streamlit/config.toml")
shutil.make_archive("pjm_project", "zip", "pjm_project")
try:
    from google.colab import files
    files.download("pjm_project.zip")
except ImportError:
    print("Saved pjm_project.zip")

## 11. (Optional) Show the Streamlit app live from Colab
Use this if you want to demo the app straight from Colab. For your Mac, unzip `pjm_project.zip` and follow `README.md` instead.
The first time, Colab prints an **IP address - paste it in the "Tunnel Password" box** that localtunnel shows.

In [ ]:
!pip -q install streamlit
!npm install -g localtunnel > /dev/null 2>&1
import urllib
print("Tunnel password (your public IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!nohup streamlit run app.py --server.port 8501 > streamlit_log.txt 2>&1 &
!sleep 5 && npx localtunnel --port 8501

## 12. Conclusion
* Cleaned a raw hourly series that was unsorted and contained duplicate, missing and impossible values.
* EDA confirmed strong daily (24 h), weekly (168 h) and yearly seasonality.
* Lag, rolling and calendar/holiday features let the tree models (XGBoost, Random Forest) reach R² ≈ 0.99 for one-hour-ahead prediction, well ahead of the naive baselines and Linear Regression (see the comparison table for ANN and LSTM).
* A recursive 7-day back-test shows realistic multi-day accuracy and error growth with horizon, compared with Holt-Winters and seasonal-naive forecasts.
* The XGBoost model is deployed in a Streamlit app that produces 1-30 day hourly forecasts, daily summaries, back-tests against real data, and CSV downloads.

**Future scope:** add weather (temperature) as an exogenous feature, retrain regularly with new data, and use direct multi-step or probabilistic forecasting to reduce long-horizon error.